# Moon OCR Worker
Script tự động trích xuất văn bản từ ảnh trong Google Drive
---
> **Trước khi chạy:** Bấm vào mũi tên ▼ góc phải → chọn **Additional connection options** → **Change runtime type** → Hardware accelerator chọn **T4 GPU**.
**Cách dùng:** Chạy từng cell từ trên xuống dưới.

## Bước 1: Mount Google Drive

Khi chạy cell bên dưới, bạn sẽ thấy thông báo:

> *Permit this notebook to access your Google Drive files?*
> *This notebook is requesting access to your Google Drive files. Granting access to Google Drive will permit code executed in the notebook to modify files in your Google Drive. Make sure to review notebook code prior to allowing this access.*

→ Chọn **Connect to Google Drive** → chọn tài khoản → **Tiếp tục**.
→ Ở màn hình quyền, chọn **Chọn tất cả** (để cấp quyền truy cập Google Drive cho notebook) → **Tiếp tục** → làm theo hướng dẫn để hoàn tất.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import numpy as np
from pathlib import Path

OCR_FOLDER = '/content/drive/MyDrive/OCR_Uploads'
os.makedirs(OCR_FOLDER, exist_ok=True)
print(f'Thu muc lam viec: {OCR_FOLDER}')

## Bước 2: Cài đặt thư viện

In [ ]:
!pip install easyocr -q
!pip install opencv-python-headless -q
!pip install python-docx -q

import easyocr
print('EasyOCR da san sang')

## Bước 3: Khởi tạo EasyOCR Reader
Hỗ trợ tiếng Anh (`en`) và tiếng Việt (`vi`), bật GPU.

In [ ]:
reader = easyocr.Reader(['en', 'vi'], gpu=True)
print('EasyOCR Reader da san sang (GPU enabled)')

## Bước 4: Hàm tiền xử lý ảnh
Chuyển sang ảnh xám + tăng độ tương phản + khử nhiễu.

In [ ]:
def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    denoised = cv2.fastNlMeansDenoising(enhanced, h=10)
    return denoised

print('Ham preprocess da san sang')

## Bước 5: Hàm OCR chính

In [ ]:
def ocr_image(image_path):
    processed = preprocess_image(image_path)
    if processed is None:
        return ''
    result = reader.readtext(processed, paragraph=True)
    lines = [item[1] for item in result]
    return '\n'.join(lines)

print('Ham OCR da san sang')

## Bước 6: Chọn format xuất file
Sửa dòng `EXPORT_FORMAT = "docx"` nếu muốn đổi:
- `"docx"` — file Word (.docx)
- `"txt"` — file văn bản (.txt)
- `"both"` — cả hai

In [ ]:
EXPORT_FORMAT = "docx"  # "txt" | "docx" | "both"
print(f'Format xuat file: {EXPORT_FORMAT}')

from docx import Document
from docx.shared import Pt, Cm
from docx.oxml.ns import qn

def save_as_docx(text, path):
    doc = Document()
    style = doc.styles['Normal']
    font = style.font
    font.name = 'Arial'
    font.size = Pt(12)
    rPr = style.element.get_or_add_rPr()
    rFonts = rPr.makeelement(qn('w:rFonts'), {
        qn('w:ascii'): 'Arial',
        qn('w:hAnsi'): 'Arial',
        qn('w:eastAsia'): 'Arial',
        qn('w:cs'): 'Arial',
    })
    rPr.append(rFonts)
    for line in text.split('\n'):
        p = doc.add_paragraph(line)
        for run in p.runs:
            run.font.name = 'Arial'
    doc.save(path)

print('Ham save_as_docx da san sang')

## Bước 7: Quét và xử lý tất cả ảnh
- Tìm các file `.jpg`, `.jpeg`, `.png`, `.webp` trong thư mục.
- Nếu file `.docx` (hoặc `.txt`) tương ứng đã tồn tại → bỏ qua.
- Chạy OCR và lưu kết quả ra file đã chọn.

In [ ]:
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}

folder = Path(OCR_FOLDER)
image_files = [f for f in folder.iterdir()
               if f.suffix.lower() in SUPPORTED_EXTENSIONS and f.is_file()]
image_files.sort()

print(f'Tong cong {len(image_files)} anh trong thu muc')
print('---')

processed_count = 0
skipped_count = 0

for img_path in image_files:
    txt_path = img_path.with_suffix('.txt')
    docx_path = img_path.with_suffix('.docx')

    # Kiem tra neu file da ton tai -> bo qua
    txt_exists = txt_path.exists()
    docx_exists = docx_path.exists()

    if EXPORT_FORMAT == 'txt' and txt_exists:
        print(f'[BO QUA] {img_path.name} - da co file txt')
        skipped_count += 1
        continue
    if EXPORT_FORMAT == 'docx' and docx_exists:
        print(f'[BO QUA] {img_path.name} - da co file docx')
        skipped_count += 1
        continue
    if EXPORT_FORMAT == 'both' and txt_exists and docx_exists:
        print(f'[BO QUA] {img_path.name} - da co ca txt va docx')
        skipped_count += 1
        continue

    print(f'[DANG XU LY] {img_path.name}...')
    try:
        text = ocr_image(str(img_path))

        if EXPORT_FORMAT in ('txt', 'both'):
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(text)
            print(f'[TXT] {txt_path.name}')

        if EXPORT_FORMAT in ('docx', 'both'):
            save_as_docx(text, str(docx_path))
            print(f'[DOCX] {docx_path.name}')

        processed_count += 1
    except Exception as e:
        print(f'[LOI] {img_path.name}: {e}')

print('---')
print(f'KET THUC: Da xu ly {processed_count}, da bo qua {skipped_count}')